# PaySight

In [55]:
import pandas as pd

In [56]:
df=pd.read_csv("Data.csv")

In [57]:
df.shape

(6362620, 11)

In [58]:
df.columns.tolist()

['step',
 'type',
 'amount',
 'nameOrig',
 'oldbalanceOrg',
 'newbalanceOrig',
 'nameDest',
 'oldbalanceDest',
 'newbalanceDest',
 'isFraud',
 'isFlaggedFraud']

In [59]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6362620 entries, 0 to 6362619
Data columns (total 11 columns):
 #   Column          Dtype  
---  ------          -----  
 0   step            int64  
 1   type            object 
 2   amount          float64
 3   nameOrig        object 
 4   oldbalanceOrg   float64
 5   newbalanceOrig  float64
 6   nameDest        object 
 7   oldbalanceDest  float64
 8   newbalanceDest  float64
 9   isFraud         int64  
 10  isFlaggedFraud  int64  
dtypes: float64(5), int64(3), object(3)
memory usage: 534.0+ MB


In [61]:
df.isnull().sum()

step              0
type              0
amount            0
nameOrig          0
oldbalanceOrg     0
newbalanceOrig    0
nameDest          0
oldbalanceDest    0
newbalanceDest    0
isFraud           0
isFlaggedFraud    0
dtype: int64

In [62]:
df.duplicated().sum()

0

In [8]:
df['type'].unique()

array(['PAYMENT', 'TRANSFER', 'CASH_OUT', 'DEBIT', 'CASH_IN'],
      dtype=object)

**Result:** 0 nulls, 0 duplicates, all 5 payment types are valid (PAYMENT, TRANSFER, CASH_OUT, CASH_IN, DEBIT). Data was already clean on this front  no rows dropped here.

## rename columns

In [63]:
df = df.rename(columns={
    'step': 'hour_number',
    'type': 'payment_type',
    'amount': 'money_amount',
    'nameOrig': 'sender_id',
    'oldbalanceOrg': 'sender_balance_before',
    'newbalanceOrig': 'sender_balance_after',
    'nameDest': 'receiver_id',
    'oldbalanceDest': 'receiver_balance_before',
    'newbalanceDest': 'receiver_balance_after',
    'isFraud': 'is_fraud',
    'isFlaggedFraud': 'system_caught_it',
})
df.columns.tolist()

['hour_number',
 'payment_type',
 'money_amount',
 'sender_id',
 'sender_balance_before',
 'sender_balance_after',
 'receiver_id',
 'receiver_balance_before',
 'receiver_balance_after',
 'is_fraud',
 'system_caught_it']

## Check: negative amounts
**Why:** money can't logically be negative a transaction amount below zero doesn't make sense

In [64]:
(df['money_amount'] < 0).sum()

0

## Check: negative balances
**Why:** same logic  no account should have a below-zero balance. Checks all 4 balance columns (sender before/after, receiver before/after) at once.

In [65]:
((df['sender_balance_before'] < 0) | (df['sender_balance_after'] < 0) |
 (df['receiver_balance_before'] < 0) | (df['receiver_balance_after'] < 0)).sum()

0

## Checking if sender's balance math adds up

**Rule:** sender's balance before, minus the amount sent, should equal sender's balance after. If it doesn't, something's off — could be a data glitch or a fraud pattern.


In [66]:
diff = (df['sender_balance_before'] - df['money_amount'] - df['sender_balance_after']).abs()
df['balance_doesnt_match'] = (diff > 0.01).astype(int)
df['balance_doesnt_match'].sum()

5077691

In [67]:
df['balance_doesnt_match'].mean() * 100

79.80503314672258

Nearly 80% of all rows show mismatched balance math. Too high to trust as a fraud signal on its own — need to check if it's spread evenly or concentrated somewhere.

In [71]:
df.groupby('payment_type')['balance_doesnt_match'].mean() * 100

payment_type
CASH_IN     100.000000
CASH_OUT     88.995263
DEBIT        30.078200
PAYMENT      54.189947
TRANSFER     95.472585
Name: balance_doesnt_match, dtype: float64

In [72]:
df.groupby('is_fraud')['balance_doesnt_match'].mean() * 100

is_fraud
0    79.907472
1     0.547912
Name: balance_doesnt_match, dtype: float64

## Building the real fraud-risk signal
Since clean balance math combined with a risky payment type is the actual pattern (not mismatched math), building one feature that combines both.

In [73]:
df['balance_matches'] = 1 - df['balance_doesnt_match']

In [74]:
df['fraud_risk_flag'] = (
    (df['balance_matches'] == 1) & (df['payment_type'].isin(['TRANSFER', 'CASH_OUT']))
).astype(int)

In [75]:
df.groupby('is_fraud')['fraud_risk_flag'].mean() * 100

is_fraud
0     4.126113
1    99.452088
Name: fraud_risk_flag, dtype: float64

### Where fraud actually occurs

In [76]:
df.groupby('payment_type')['is_fraud'].sum()

payment_type
CASH_IN        0
CASH_OUT    4116
DEBIT          0
PAYMENT        0
TRANSFER    4097
Name: is_fraud, dtype: int64

Fraud only ever happens in TRANSFER and CASH_OUT transactions in this dataset — CASH_IN, DEBIT, and PAYMENT never carry fraud. Useful scope for the ML model in a later stage.

## Checking the receiver side too
Ran the same balance-reconciliation logic on the receiver's balance, to see if it's worth keeping as a second signal.

In [77]:
diff_dest = (df['receiver_balance_before'] + df['money_amount'] - df['receiver_balance_after']).abs()
df['dest_balance_doesnt_match'] = (diff_dest > 0.01).astype(int)
df.groupby('is_fraud')['dest_balance_doesnt_match'].mean() * 100

is_fraud
0    65.844177
1    56.495799
Name: dest_balance_doesnt_match, dtype: float64

**Result:** ~66% for legit vs ~56% for fraud — barely any separation, unlike the sender-side check (80% vs 0.5%). Makes sense: fraud drains the *sender's* account cleanly, but the receiver side often includes merchant accounts that PaySim doesn't track balances for properly, so receiver math stays noisy regardless of fraud. **Not a useful signal — dropping this column, not adding clutter to the final dataset.**

In [78]:
df = df.drop(columns=['dest_balance_doesnt_match'])
df.columns.tolist()

['hour_number',
 'payment_type',
 'money_amount',
 'sender_id',
 'sender_balance_before',
 'sender_balance_after',
 'receiver_id',
 'receiver_balance_before',
 'receiver_balance_after',
 'is_fraud',
 'system_caught_it',
 'balance_doesnt_match',
 'balance_matches',
 'fraud_risk_flag']

In [79]:
summary = {
    "row_count": len(df),
    "total_amount": round(df['money_amount'].sum(), 2),
    "fraud_count": int(df['is_fraud'].sum()),
    "system_caught_it_count": int(df['system_caught_it'].sum()),
    "fraud_risk_flag_count": int(df['fraud_risk_flag'].sum())
}
summary

{'row_count': 6362620,
 'total_amount': 1144392944759.77,
 'fraud_count': 8213,
 'system_caught_it_count': 16,
 'fraud_risk_flag_count': 270358}

In [80]:
df.to_csv("cleaned_transactions.csv", index=False)

## Key takeaways
- Data was clean on nulls, duplicates, and negative values — no issues there.
- Raw balance-mismatch rate (~80%) looked like a fraud signal at first glance but was misleading — fraud actually shows up as *clean*, reconciled balance math, not broken math.
- Built `fraud_risk_flag`: clean balance math + TRANSFER/CASH_OUT type — catches 99.45% of real fraud vs only ~4% false-positive rate on legit transactions.
- The company's existing fraud system (`system_caught_it`) catches only 16 of 8,213 real fraud cases — about 0.2%. This is the core business problem the rest of this project addresses.
- Fraud only ever occurs in TRANSFER and CASH_OUT transactions.